In [2]:
from qiskit import QuantumCircuit
from qiskit.primitives import Sampler
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

def create_bit_flip_code_circuit(apply_error=False, error_qubit=0):
    qc = QuantumCircuit(5, 1)

    # Encode: |ψ⟩ = (|0⟩ + |1⟩)/√2 → (|000⟩ + |111⟩)/√2
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(0, 2)

    # Simulate a bit-flip error
    if apply_error:
        qc.x(error_qubit)

    # Syndrome detection
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)

    # Correction
    qc.ccx(3, 4, 1)
    qc.cx(3, 0)
    qc.cx(4, 2)

    # Decode
    qc.cx(0, 1)
    qc.cx(0, 2)

    # Measure the logical qubit
    qc.measure(0, 0)
    return qc

# Create circuits
qc_clean = create_bit_flip_code_circuit()
qc_error = create_bit_flip_code_circuit(apply_error=True, error_qubit=1)

# Use Qiskit Sampler
sampler = Sampler()
job_clean = sampler.run(qc_clean)
job_error = sampler.run(qc_error)

# Get results
result_clean = job_clean.result().quasi_dists[0]
result_error = job_error.result().quasi_dists[0]

# Convert to histogram format (only measuring 1 qubit, so 0 and 1)
counts_clean = {str(k): int(v * 1024) for k, v in result_clean.items()}
counts_error = {str(k): int(v * 1024) for k, v in result_error.items()}

# Plot
print("Clean:", counts_clean)
print("With error on qubit 1:", counts_error)
plot_histogram([counts_clean, counts_error], legend=['Clean', 'Error on Q1'])
plt.show()


Clean: {'0': 511, '1': 511}
With error on qubit 1: {'0': 511, '1': 511}


/var/folders/yk/bmjk7mhn3kl8z2nqqk4swv_m0000gn/T/ipykernel_62419/3826670511.py:42: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
